<a href="https://colab.research.google.com/github/kscaman/MAP583/blob/main/TP/TP07_Language_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP 8 - Language Modeling

Taken from Kevin Scaman and David Robin course MAP583 of Ecole Polytechnique.

- Part A is a short introduction to language modeling with Markov chains.
- Part B is the most interesting, and the goal of this session, adjust your time budget accordingly.

In [343]:
# !pip install --quiet "git+https://gitlab.com/robindar/dl-scaman_checker.git"
# from dl_scaman_checker import TP07
# check_install()

Define utility functions

In [ ]:
import torch
import numpy as np
from urllib.request import urlretrieve

def printable_parameter_count(model):
    count = sum(p.numel() for p in model.parameters())
    return human_readable_unit(count)

def human_readable_unit(integer):
    assert integer >= 0, f"Cannot compute human-readable form of negative integer '{integer}'"
    category = int(np.floor(np.log10(integer) / 3))
    cfg = [ ("", 0), ("K", 0), ("M", 1), ("G", 1), ("T", 2), ("P", 2) ]
    trail, rnd = cfg[category]
    return f"{np.round(integer / 10 ** (3*category), rnd):.{rnd}f}{trail}"

def download_tinyshakespeare(filename):
    url = (
        "https://raw.githubusercontent.com/"
        "karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    )
    path, headers = urlretrieve(url, filename)
    ETag = '"a82bf4e8979c373a24f616ef6c044f821e18ce64322e5e1280f069f2910b3653"'
    sig = ""
    if ("ETag" in headers) and (headers["ETag"] == ETag):
        sig = ", ETag matches expectation"
    return f"Download ok{sig}. File available at '{filename}'"


In [345]:
filename = "input.txt"
download_tinyshakespeare(filename)

"Download ok, ETag matches expectation. File available at 'input.txt'"

In [346]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Part A - Language modeling basics

Our goal is to generate random text. To this end, we will define a probability distribution
over "texts", such that we are able to sample from this distribution, and then optimize over
such distributions.

Let $\Sigma$ be a set, called the "alphabet".
A "text" of size $k \in \mathbb{N}$ is an element of $\Sigma^k$.
Define $X : \mathbb{N} \to \Sigma$ a sequence of random variables with a given distribution.
We model the probability of a text $t \in \Sigma^k$
as $$\mathbb{P}(t) = \mathbb{P}(X_{\leq k} = t) = \prod_{i \in [k]} \mathbb{P}(X_i = t_i \mid X_{< i} = t_{< i}) $$
For every distribution on sequences, this yields a simple way to sample: one character at a time.

We will represent these conditional distributions as a function
$f_i : \Sigma^i \to \mathbb{R}^\Sigma$ for $i \in \mathbb{N}$,
such that $$\mathbb{P}(X_i = s \,|\, X_{<i} = q) = \frac{ \exp\left(r_s\right) }{ \sum_{j \in \Sigma} \exp\left(r_j\right) } \quad\text{where}\quad r = f_i(q)$$
This transformation is implemented as `F.softmax` in PyTorch. The variable `r` is called the "logit".
Usually the function $f$ only depends on a finite number of previous characters, called the "block size" of the model.
For instance, a Markov chain has a block size of 1. This influences the computational cost of the model, but also the size of the context that it can take into account.

In [797]:
class Alphabet:
    def __init__(self, alphabet):
        self.characters = alphabet

        self.string_to_int = { ch: i for i,ch in enumerate(alphabet) }
        self.int_to_string = { i: ch for i,ch in enumerate(alphabet) }

    def encode(self, s):
        # return torch.tensor([ self.stoi[c] for c in s ], dtype=torch.long)
        return [ self.string_to_int[c] for c in s ]

    def decode(self, l):
        if isinstance(l, torch.Tensor):
            l = l.detach().numpy()
        return "".join([ self.int_to_string[i] for i in list(l) ])

In [798]:
alphabet = Alphabet("\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz")

In [799]:
alphabet.encode("Lorem ipsum")

[24, 53, 56, 43, 51, 1, 47, 54, 57, 59, 51]

In [800]:
alphabet.decode([ 24, 53, 56, 43, 51 ])

'Lorem'

### A.1 - Simple Bigram Model

For our first custom model, we represent text with a Markov chain over the alphabet,  
where the probability of each character in the text depends only on the previous character.

In [801]:
class BigramLogitModel():
    """A simple Bigram Model for the limited alphabet 'HELO '."""
    def __init__(self):
        self.alphabet = Alphabet("HELO ")
        # p = Transition Matrix P(next_char | current_char)
        self.transition_matrix = np.array(
            #    H  , E  , L  , O  ,space
            [
                [0.1, 0.6, 0.1, 0.1, 0.1], # H
                [0.1, 0.1, 0.6, 0.1, 0.1], # E
                [0.1, 0.1, 0.4, 0.3, 0.1], # L
                [0.1, 0.1, 0.1, 0.1, 0.6], # O
                [0.6, 0.1, 0.1, 0.1, 0.1], # space
            ],
            dtype=np.float32,
        )

    def __call__(self, seq):
        last_char = seq[-1]
        p = self.transition_matrix[last_char]
        logits = np.log(p)
        return logits

simple_model = BigramLogitModel()

In [802]:
small_alphabet = simple_model.alphabet
preprompt = small_alphabet.encode("HE")
simple_model(preprompt)

array([-2.3025851, -2.3025851, -0.5108256, -2.3025851, -2.3025851],
      dtype=float32)

Notice that the bigram model only takes the last character of the prompt into account. The output of the model is the same for any preprompt ending with C

In [803]:
simple_model(small_alphabet.encode("LE"))

array([-2.3025851, -2.3025851, -0.5108256, -2.3025851, -2.3025851],
      dtype=float32)

In [804]:
simple_model(small_alphabet.encode("EE"))

array([-2.3025851, -2.3025851, -0.5108256, -2.3025851, -2.3025851],
      dtype=float32)

Recall that the softmax function allows us to get back to a probability distribution from a tensor of logits

In [805]:
logits = simple_model(small_alphabet.encode("H"))
F.softmax(torch.tensor(logits), dim=-1)

tensor([0.1000, 0.6000, 0.1000, 0.1000, 0.1000])

Write the `prompt` function, which takes a logit-computation function $f$, a number of characters to sample, and a preprompt to initialize the sequence. Additionally, take a temperature parameter $T \in \mathbb{R}_+^*$, and sample from the logits $s \mapsto \frac{1}{T} f(s)$, such that in the limit $T \to +\infty$ it samples from the uniform distribution.

You can use the functions `F.softmax` and `torch.multinomial(p, num_samples=1)` for sampling.

In [806]:
def prompt(model, num_chars, preprompt: str, temp: float, alphabet: Alphabet):
    for n in range(num_chars):
        logits = model(alphabet.encode(preprompt))
        probabilities = F.softmax(1/temp*torch.tensor(logits), dim=-1)
        character = torch.multinomial(probabilities, num_samples=1)
        preprompt += alphabet.decode(character)
    return preprompt

prompt(simple_model, 100, preprompt="HEL", temp=0.5, alphabet=small_alphabet)

'HELLLLLLLLLO HELLHEL HEO HELLO HELLLLLLO HELLO HEO O HELLLO HELEO HELLLLO HEHELOO HELLO HELHELO HELO HE'

What do you notice when you decrease the temperature ?

In [807]:
prompt(simple_model, 10, preprompt="H", temp=10, alphabet=small_alphabet)

'HEE O LEHE '

### A.2 Language model on Shakespeare data

The transition matrix of a language model is obtained by parsing large databases of text.  
Here we use a corpus of Shakespeare and compute the empirical frequency of groups of n letters (n-grams).

In [808]:
def read_from_file(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        text = f.read()

    chars = "".join(sorted(list(set(text))))
    alphabet = Alphabet(chars)

    data = torch.tensor(alphabet.encode(text), dtype=torch.long)
    n = int(0.9 * len(data))
    train_data = data[:n]
    val_data = data[n:]

    return alphabet, train_data, val_data

In [809]:
alphabet, train_data, val_data = read_from_file(filename)
vocab_size = len(alphabet.characters)
alphabet.characters, vocab_size

("\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz", 65)

In [810]:
train_data.shape, train_data.dtype

(torch.Size([1003854]), torch.int64)

In [811]:
train_size, val_size = [ human_readable_unit(len(d)) for d in (train_data, val_data) ]
print(f"Training: {train_size} chars, Validation: {val_size} chars")

Training: 1.0M chars, Validation: 112K chars


In [ ]:
class NGramModel:
    def __init__(self, alphabet, dataset, n=4):
        vocab_size = len(alphabet.characters)
        logits = np.ones(tuple([vocab_size] * (n + 1)), dtype=np.float32) * 1e-6
        for i in range(len(dataset) - n):
            logits[tuple(dataset[i : i + n + 1])] += 1
        self.logits = np.log(logits)
        self.histsize = n

    def __call__(self, seq):
        assert len(seq) >= self.histsize
        return self.logits[tuple(seq[-self.histsize :])]

ngram_model = NGramModel(
    alphabet=alphabet,
    dataset=train_data.numpy(),
    n=3,
)

In [860]:
output = prompt(ngram_model, 100, preprompt="JUL", temp=0.5, alphabet=alphabet)
print(output)

JULIET:
O, the poing made is a come the put affect all he as you with me shall be no shall have more as


## Part B - Language Modeling with Transformers

### B.1 - Training setup with bigram models

To prepare the setup for training language models by deep learning techniques, but before diving into the details of the model, let us re-build the bigram model in a learning-friendly manner,
and learn the transition probability matrix to maximize the log-likelihood of the training data

We provide the `get_batch` and `train_model` functions.
We will compute predictions for all elements of the batch and for all timesteps at once, to speed up training.
This means for a batch size of 1 and an alphabet of size 65, the prediction for `xb.shape = (1, block_size)`
should have `model(xb).shape = (1, block_size, 65)`.
Read carefully the shape assertion.

In [360]:
def get_batch(data, batch_size = 16, block_size = 32):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [361]:
def train_model(model, optimizer, iterations=1):
    for _ in range(iterations):
        xb, yb = get_batch(train_data)
        logits = model(xb)
        assert logits.shape == (*yb.shape, len(alphabet.characters))
        logits = logits.view(-1, logits.shape[-1])
        loss = F.cross_entropy(logits, yb.view(-1))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    return model, optimizer

In [362]:
@torch.no_grad()
def estimate_negloglikelihood(model, data, eval_iters=250):
    model.eval()
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
        X, Y = get_batch(data)
        logits = model(X)
        logits = logits.view(-1, logits.shape[-1])
        loss = F.cross_entropy(logits, Y.view(-1))
        losses[k] = loss.item()
    model.train()
    return losses.mean()

Write the Bigram model as a trainable `torch.nn.Module`. Make sure it works with batches of shape `(batch_size, block_size)`, for which the forward function should return a tensor of shape `(batch_size, block_size, alphabet_size)`.

In [430]:
class Bigram(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Token embedding table - each token gets mapped to a vector of logits
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, idx):
        # idx shape: (B, T) where B=batch_size, T=block_size
        # For bigram model, we use the embedding of each token to predict the next token
        logits = self.token_embedding_table(idx)  # (B, T, vocab_size)
        return logits

In [431]:
bigram_model = Bigram(len(alphabet.characters)).to(device)
print(f"Bigram model has {printable_parameter_count(bigram_model)} parameters")
optimizer = torch.optim.Adam(bigram_model.parameters(), lr=1e-2)

Bigram model has 4K parameters


In [433]:
for iter in range(100):
    bigram_model, optimizer = train_model(bigram_model, optimizer, iterations=100)
    train_loss, val_loss = [ estimate_negloglikelihood(bigram_model, data) for data in [ train_data, val_data ] ]
    print(f"train loss {train_loss:.4f}, val loss {val_loss:.4f}")

train loss 2.5213, val loss 2.5423
train loss 2.5043, val loss 2.5221
train loss 2.5043, val loss 2.5221
train loss 2.4891, val loss 2.5145
train loss 2.4891, val loss 2.5145
train loss 2.4807, val loss 2.5121
train loss 2.4807, val loss 2.5121
train loss 2.4780, val loss 2.5052
train loss 2.4780, val loss 2.5052
train loss 2.4781, val loss 2.4988
train loss 2.4781, val loss 2.4988
train loss 2.4694, val loss 2.4979
train loss 2.4694, val loss 2.4979
train loss 2.4692, val loss 2.4907
train loss 2.4692, val loss 2.4907
train loss 2.4657, val loss 2.4953
train loss 2.4657, val loss 2.4953
train loss 2.4670, val loss 2.4877
train loss 2.4670, val loss 2.4877
train loss 2.4656, val loss 2.4946
train loss 2.4656, val loss 2.4946
train loss 2.4658, val loss 2.4876
train loss 2.4658, val loss 2.4876
train loss 2.4615, val loss 2.4907
train loss 2.4615, val loss 2.4907
train loss 2.4598, val loss 2.4891
train loss 2.4598, val loss 2.4891
train loss 2.4610, val loss 2.4881
train loss 2.4610, v

In [724]:
def bigram_wrapper(input_tokens):
    tokens_torch = torch.LongTensor(input_tokens).unsqueeze(0).to(device)
    logits = bigram_model(tokens_torch)
    return logits[0, -1, :].detach().cpu().numpy()

output = prompt(bigram_wrapper, 50, preprompt="A", temp=0.5, alphabet=alphabet)
print(output)

And be mathisuthalathe oorngee wincoth y ave thime 


The result is quite disappointing, but the bigram model is also clearly an oversimplification of language, it would be very surprising if it could just spit out entire paragraphs from Shakespeare.
To assess performance in an easier setting, prompt this model with `"Jul"`, and see how often you can get it to auto-complete `"Juliet"` (a name which appears a lot in Shakespeare's plays). What happens if you lower the temperature too much ? How is this different from what happens when you prompt with `"the "` ?

In [664]:
for _ in range(50):
    output = prompt(bigram_wrapper, 3, preprompt="Jul", temp=0.5, alphabet=alphabet)
    print(output)

Jul be
Jul at
Julo b
Jul d 
Jullea
Julde 
Julour
Jul my
Jul th
Juller
Julore
Jullo 
Julat 
Julall
Jullis
Juld m
Jullle
Julou 
Jullac
Julath
Jullim
Julis 
Jule t
Juleso
Jul be
Juled 
Juline
Julo s
Jul t 
Juld t
Julare
Juleri
Juldic
Juld t
Jul he
Julire
July t
Jul th
Jule t
Julire
Jule a
Jululi
Juld s
Jullle
Jul il
Jule t
Jular 
Jul he
Jule a
Juld i


### B.2 - Advanced Language Models

Now let us build a GPT-like architecture.
We will use the causal / masked attention operation,
where the mask $M \in \{ 0 ,  1\}^{d \times d}$ is defined as $M_{i,j} = 1$ if and only if $i > j$.
$$ \operatorname{Attention} : X \in \mathbb{R}^{d \times d} \mapsto \left[ \dfrac{M_{i,j} \exp(X_{i,j}) }{ \sum_{k} M_{i,k} \exp(X_{i,k}) } \right]_{i,j} \in \mathbb{R}^{d \times d} $$
If you want use `F.softmax`, you may want to use the functions `Tensor.masked_fill` and `torch.tril` as well.
You can implement matrix multiplication with a learnable weight with `torch.nn.Linear(bias=false)`.

We write $B$ for the batch size, $T$ for the block size (or "timesteps"), and $C$ for the embedding dimension (or number of "channels").

1. Write a SelfAttention module implementing the operation with weights $K \in \mathbb{R}^{C \times d}$, $Q \in \mathbb{R}^{C \times d}$ and $V \in \mathbb{R}^{C \times d}$
$$ X \in \mathbb{R}^{B \times T \times C} \mapsto \left[ \operatorname{Attention}\left[\frac{(X_i \cdot K) \cdot (X_i \cdot Q)^T}{\sqrt{d}}\right] \cdot (X_i \cdot V) \right]_{i \in [B]} $$
We choose the variable `d` such that `d * num_heads = n_embd`, so the size is unchanged after going through a multi-head block.

3. Write a Multi-Head attention module computing the output of `num_heads` independent Self-Attention modules, then concatenating the result.

4. Write a Transformer module, computing the composition of a residual block with a multi-head attention module as residual,
    followed by a residual block with a two-layer ReLU-network for the residual. Add a layer norm at the start of each residual branch to avoid training instabilities.

In [ ]:
class MaskedSelfAttention(nn.Module):
    def __init__(self, head_size, n_embd, block_size):
        super().__init__()
        self.head_size = head_size
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        
        # Register the mask as a buffer (not a parameter)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
    
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)
        v = self.value(x) # (B, T, head_size)
        
        # Compute attention scores
        wei = q @ k.transpose(-2, -1) * (self.head_size ** -0.5)  # (B, T, T)
        
        # Apply causal mask
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        
        # Apply attention to values
        out = wei @ v  # (B, T, head_size)
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, n_embd, block_size):
        super().__init__()
        self.heads = nn.ModuleList([
            MaskedSelfAttention(head_size, n_embd, block_size) 
            for _ in range(num_heads)
        ])
        self.proj = nn.Linear(n_embd, n_embd)
    
    def forward(self, x):
        # Concatenate outputs from all heads
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        # Apply final projection
        out = self.proj(out)
        return out

class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, block_size, hidden_size):
        super().__init__()
        head_size = n_embd // n_head
        self.self_attention = MultiHeadAttention(n_head, head_size, n_embd, block_size)
        self.ffwd = nn.Sequential(
            nn.Linear(n_embd, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, n_embd),
        )
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    
    def forward(self, x):
        # Self-attention with residual connection and layer norm
        x = x + self.self_attention(self.ln1(x))
        # Feed-forward with residual connection and layer norm
        x = x + self.ffwd(self.ln2(x))
        return x

In [861]:
class LanguageModel(nn.Module):
    def __init__(self, n_embd, n_head, block_size, hidden_size, n_layers, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        blocks = [
            TransformerBlock(n_embd=n_embd,
                             n_head=n_head,
                             block_size=block_size,
                             hidden_size=hidden_size)
            for _ in range(n_layers) ]
        self.blocks = nn.Sequential(*blocks)
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self._block_size = block_size

    def block_size(self):
        return self._block_size

    def forward(self, idx):
        B, T = idx.shape
        # idx is a (B,T)-shaped LongTensor
        tok_emb = self.token_embedding_table(idx) # shape: (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # shape: (T,C)
        x = tok_emb + pos_emb # shape: (B,T,C)
        x = self.blocks(x) # shape: (B,T,C)
        x = self.ln_f(x) # shape: (B,T,C)
        logits = self.lm_head(x) # shape: (B,T,vocab_size)
        return logits

---

Initialize your language model, and train it repeatedly or tweak the hyperparameters and retrain, until you reach training losses below 1.5 nats/char. Prompt regularly during training to witness the evolution of the generated samples, but don't waste your time budget sampling too much if your training loss is above 1.75, you would get mostly unintelligible gibberish.

You should be able, by the end of the session, to get didaskalia in capitals with real Shakespeare character names followed by a colon, and vaguely old-english-sounding sentences separated by dots.

In [862]:
config = {
    "n_head": 4,
    "n_embd": 64,
    "hidden_size": 1024,
    "n_layers": 2,
    "block_size": 32,
    "vocab_size": len(alphabet.characters),
}
language_model = LanguageModel(**config).to(device)
paramcount = printable_parameter_count(language_model)
print(f"Language model has {paramcount} parameters")

optimizer = torch.optim.AdamW(language_model.parameters(), lr=1e-3)
total_iter = 0

Language model has 308K parameters


In [863]:
max_iters, eval_interval = 1000, 100

for iter in range(max_iters // eval_interval):
    language_model, optimizer = train_model(language_model, optimizer, iterations=eval_interval)
    train_loss, val_loss = [ estimate_negloglikelihood(language_model, data) for data in [ train_data, val_data ] ]
    total_iter += eval_interval
    print(f"step {total_iter:4d}: train loss {train_loss:.4f}, val loss {val_loss:.4f}")

step  100: train loss 2.6344, val loss 2.6330
step  200: train loss 2.4915, val loss 2.4995
step  300: train loss 2.4002, val loss 2.4001
step  400: train loss 2.3182, val loss 2.3236
step  500: train loss 2.2598, val loss 2.2866
step  600: train loss 2.2120, val loss 2.2357
step  700: train loss 2.1754, val loss 2.2022
step  800: train loss 2.1190, val loss 2.1657
step  900: train loss 2.0999, val loss 2.1366
step 1000: train loss 2.0699, val loss 2.1246


In [789]:
def lm_wrapper(input_tokens):
    input_window = input_tokens[-language_model.block_size():]
    tokens_torch = torch.LongTensor(input_window).unsqueeze(0)
    return language_model(tokens_torch)[0,-1,:].detach().cpu().numpy()

output = prompt(lm_wrapper, 200, preprompt="DUKE", temp=1.0, alphabet=alphabet)
print(output)

DUKE VINCENRY BUCH:
That thy doichisf thir
thou him love granius.N for of fair bovest that he chowleriance we is prait's whilseed
e would to miffice argies, here hold: rue shalt, a my dom ony me,
Or in is


---